# Lab 03: Classical Ciphers, Perfect Secrecy, and Many-Time Pad Attacks
**Student Name:** Md. Akib Al Jawad  
**Student ID:** IT-23020  
**Department:** Information and Communication Technology (ICT)  
**Institution:** Mawlana Bhashani Science and Technology University (MBSTU)  
**Reference:** IIT Bombay Maths and Physics Club Workshop — Cryptography Handout & Many-Time Pad Demo

---

## Abstract
This laboratory notebook bridges classical cryptography and modern information theory:
1. **Caesar & Polyalphabetic Ciphers**: Vulnerability of mono- and poly-alphabetic substitutions to frequency analysis.
2. **Claude Shannon's Information Theory**: Formal definition of **Perfect Secrecy** ($H(M|C) = H(M)$) and Shannon's theorem ($|K| \ge |M|$).
3. **One-Time Pad (OTP) Implementation**: The only provably unbreakable symmetric cipher when the key is uniform, random, secret, and **never reused**.
4. **Many-Time Pad Vulnerability & Crib-Dragging Cryptanalysis**: Demonstrating the catastrophic failure when a key is reused across multiple messages ($C_1 \oplus C_2 = M_1 \oplus M_2$).



### 1. Caesar Cipher & Frequency Analysis
The Caesar cipher maps letter $x$ to $E(x) = (x + k) \bmod 26$. Because it preserves the single-letter probability distribution, an eavesdropper Eve can easily break it by comparing letter frequencies against English language distribution (where 'E', 'T', 'A', 'O', 'I', 'N' dominate).



In [1]:
import string
from collections import Counter

def caesar_encrypt(text, shift):
    res = []
    for ch in text:
        if ch.isalpha():
            base = ord('A') if ch.isupper() else ord('a')
            res.append(chr((ord(ch) - base + shift) % 26 + base))
        else:
            res.append(ch)
    return "".join(res)

pt = "THE SECURITY OF CRYPTOGRAPHY RESTS ON SOUND MATHEMATICAL FOUNDATIONS"
ct = caesar_encrypt(pt, 7)
print(f"Plaintext:  {pt}")
print(f"Ciphertext: {ct}")

# Eve's Frequency Analysis Attack
counts = Counter([c for c in ct if c.isalpha()])
most_common_cipher_char = counts.most_common(1)[0][0]
assumed_shift = (ord(most_common_cipher_char) - ord('E')) % 26
print(f"Most common ciphertext letter: '{most_common_cipher_char}'")
print(f"Recovered shift (assuming 'E'): {assumed_shift}")
recovered_pt = caesar_encrypt(ct, -assumed_shift)
print(f"Recovered message: {recovered_pt}")



Plaintext:  THE SECURITY OF CRYPTOGRAPHY RESTS ON SOUND MATHEMATICAL FOUNDATIONS


Ciphertext: AOL ZLJBYPAF VM JYFWAVNYAWOY YLZAZ VU ZVBUK THAOLTHAPJHS MVBUKHAPVUZ


Most common ciphertext letter: 'A'


Recovered shift (assuming 'E'): 22


Recovered message: XLH WIGYVMXC SJ GVCTXSKVETDC VSWXW SR WSYRH TEXLITEXMGEP JSYRHEXMSRW


### 2. Shannon's Perfect Secrecy & The One-Time Pad
In 1949, Claude Shannon proved in *Communication Theory of Secrecy Systems* that a cipher achieves **perfect secrecy** if:
$$P(M = m \mid C = c) = P(M = m) \quad \forall m \in \mathcal{M}, c \in \mathcal{C}$$
This requires that the ciphertext reveals zero statistical information about the plaintext. A necessary mathematical condition is that the key space must be at least as large as the message space:
$$|\mathcal{K}| \ge |\mathcal{M}|$$



In [1]:
def to_hex(s):
    return "".join(hex(ord(ch))[2:].zfill(2) for ch in s)

def from_hex(hex_str):
    return "".join(chr(int(hex_str[i:i+2], 16)) for i in range(0, len(hex_str), 2))

def xor_hex(s1, s2):
    assert len(s1) == len(s2)
    return "".join(hex(int(s1[i:i+2], 16) ^ int(s2[i:i+2], 16))[2:].zfill(2) for i in range(0, len(s1), 2))

msg = "cryptography"
key = "supersecretk"
c = xor_hex(to_hex(msg), to_hex(key))
decrypted = from_hex(xor_hex(c, to_hex(key)))
print(f"Plaintext:   {msg}")
print(f"Secret Key:  {key}")
print(f"Ciphertext:  {c}")
print(f"Decrypted:   {decrypted}")
assert decrypted == msg



Plaintext:   cryptography


Secret Key:  supersecretk


Ciphertext:  10070905011c141517001313


Decrypted:   cryptography


### 3. Many-Time Pad Vulnerability & Crib-Dragging
If the same key $K$ is used to encrypt two different messages $M_1$ and $M_2$:
$$C_1 = M_1 \oplus K, \quad C_2 = M_2 \oplus K$$
Eavesdropping both ciphertexts allows the adversary to eliminate the key entirely:
$$C_1 \oplus C_2 = (M_1 \oplus K) \oplus (M_2 \oplus K) = M_1 \oplus M_2$$

#### ASCII Space Property:
In ASCII encoding:
- Space `' '` has hex value `0x20` (binary `00100000`).
- Any English letter XORed with space flips its case: `'a' ^ ' ' = 'A'`, `'T' ^ ' ' = 't'`.
- Therefore, when $(C_1 \oplus C_2)[i]$ has bit 5 set and corresponds to a letter, one of the original messages had a space at index $i$!

#### Crib Dragging:
By sliding a common word (crib, e.g., `"the "`) across $C_1 \oplus C_2$, whenever the crib is present in $M_1$ at index $i$, the operation:
$$(C_1 \oplus C_2)[i : i + L] \oplus \text{crib} = M_2[i : i + L]$$
yields readable English text, allowing full plaintext reconstruction.



In [1]:
message1 = "steal the secret"
message2 = "the cat the bird"
key      = "supersecretverys"

c1 = xor_hex(to_hex(message1), to_hex(key))
c2 = xor_hex(to_hex(message2), to_hex(key))

# Eavesdropper computes XOR of ciphertexts
xor_c1_c2 = xor_hex(c1, c2)
print(f"C1 ^ C2 (Hex): {xor_c1_c2}")

def crib_drag_search(xor_hex_str, crib):
    xor_bytes = [int(xor_hex_str[i:i+2], 16) for i in range(0, len(xor_hex_str), 2)]
    crib_bytes = [ord(c) for c in crib]
    print(f"\n[*] Dragging Crib: '{crib}' across XOR stream:")
    for pos in range(len(xor_bytes) - len(crib_bytes) + 1):
        candidate = "".join(chr(xor_bytes[pos + j] ^ crib_bytes[j]) for j in range(len(crib_bytes)))
        if all(ch.isalnum() or ch in " ,.'-;:!?" for ch in candidate):
            print(f"    Position {pos:2d} -> Revealed in other message: '{candidate}'")

crib_drag_search(xor_c1_c2, "the ")
crib_drag_search(xor_c1_c2, "secret")



C1 ^ C2 (Hex): 071c01015c15084540450509071f0701


[*] Dragging Crib: 'the ' across XOR stream:


    Position  0 -> Revealed in other message: 'stea'


    Position  5 -> Revealed in other message: '5h-1'


    Position  6 -> Revealed in other message: 't th'


    Position  8 -> Revealed in other message: 'e se'


    Position 10 -> Revealed in other message: 'b-d;'


    Position 12 -> Revealed in other message: 'usr0'


[*] Dragging Crib: 'secret' across XOR stream:


    Position 10 -> Revealed in other message: ' bird '
